# Aerodynamic Coefficients: Calculation, Verification, and Theoretical Comparison

This notebook follows the actual `lib.vlm.VLM` workflow and explains how forces, moments, and their coefficients are calculated. SI units are used throughout: lengths in m, velocity in m/s, density in kg/m³, forces in N, and moments in N·m; angles in the `VLM` object are expressed in radians. Helper functions accept degrees only where explicitly stated.

**Scope:** steady, incompressible, inviscid VLM for thin lifting surfaces, attached flow, and a prescribed wake. The model does not predict viscous drag, separation, stall, or compressibility. Numerical checks verify internal consistency; comparison with lifting-line theory is a comparison between theories, not experimental validation.

## 1. VLMPy Cases and Available Data

The reference case is a symmetric rectangular wing with no sweep or dihedral, total span $b=8$ m, chord $c=1$ m, NACA 0012 section, and $AR=8$. Freestream speed is $40$ m/s, density is $1.225$ kg/m³, $\alpha=3^\circ$, and $\beta=0$. `n` and `m` are the numbers of panels across the full span and chord, respectively.

`run_vlm()` calculates the geometry, panels, influence matrix `P_ij`, right-hand side `w_i`, circulation `gammas`, induced-velocity projections, and coefficients. Internal diagnostics are suppressed for readability; the actual data remain in the `case` object.

## Theory and How to Use the Code

A dimensionless coefficient relates a force or moment to a chosen dynamic pressure and reference geometry: $q_\infty=\tfrac12\rho U^2$, $C_F=F/(q_\infty S_{ref})$, and a moment coefficient uses an additional reference length. The direction of each component, sign convention, and moment reference point are essential to interpreting or comparing coefficients.

VLMPy solves a circulation system and provides both scalar lift/induced-drag outputs and force/moment components obtained from vector panel loads. These are not interchangeable for arbitrary orientations or sideslip. The later cells reconstruct the actual code paths and explicitly identify the assumptions behind their pressure-equivalent and wind-axis calculations.

**Usage and examples.** Run the cells in order. The baseline uses the public `VLM` class; the optional loading helper accepts trusted saved results. The calculations cover forces, sectional distributions, angle sweeps, mesh sensitivity, and an independent lifting-line comparison. Theoretical agreement is not experimental validation. See [NASA Glenn: Lift Equation](https://www1.grc.nasa.gov/beginners-guide-to-aeronautics/lift-equation/) and the detailed references in the comparison section.

In [16]:
import contextlib
import io
import sys
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def find_repository(start):
    for directory in (start, *start.parents):
        if (directory / 'src' / 'vlm').is_dir():
            return directory
    raise RuntimeError('Could not locate the repository directory src/vlm.')


ROOT = find_repository(Path.cwd().resolve())
for directory in (ROOT, ROOT / 'src' / 'vlm'):
    if str(directory) not in sys.path:
        sys.path.insert(0, str(directory))

from src.vlm.lib.vlm import VLM


def solve_case(plane, alpha_deg, n=24, m=4, speed=40.0, density=1.225, beta_deg=0.0):
    case = VLM(plane, speed, density, np.deg2rad(alpha_deg), np.deg2rad(beta_deg), n, m)
    with contextlib.redirect_stdout(io.StringIO()):
        case.run_vlm()
    return case


rectangular_wing = {'wing_sections': [{
    'span_fraction': 8.0, 'chord_root': 1.0, 'chord_tip': 1.0,
    'sweep': 0.0, 'dihedral': 0.0, 'NACA_root': '0012', 'NACA_tip': '0012',
}]}
case = solve_case(rectangular_wing, alpha_deg=3.0)
assert case.P_ij.shape == (len(case.panel_data), len(case.panel_data))
assert np.isfinite(case.gammas).all()
print(f'Baseline case: {len(case.panel_data)} panels; CL={case.CL:.6f}; CD={case.CD:.6f}')

Baseline case: 96 panels; CL=0.246735; CD=0.002682


### Loading Saved Results

The following helper accepts JSON manifests produced by `VLM.save_results`, together with their associated NPZ files. It checks the schema and configuration checksum, reconstructs the geometry and mesh from the saved parameters, and delegates array loading to `VLM.load_results`. Files must come from a trusted source. The round-trip check uses a temporary directory; to analyze several saved cases, pass their paths to `load_result_cases`.

In [17]:
import json
import tempfile


def load_result_cases(paths):
    cases = []
    for path in paths:
        manifest = json.loads(Path(path).read_text(encoding='utf-8'))
        configuration = manifest.get('configuration')
        if manifest.get('schema') != 'vlmpy.results' or manifest.get('schema_version') != 1 or not isinstance(configuration, dict):
            raise ValueError(f'Invalid results manifest: {path}')
        if VLM.configuration_checksum(configuration) != manifest.get('configuration_checksum'):
            raise ValueError(f'Invalid configuration checksum: {path}')
        flight = configuration.get('flight_parameters')
        if not isinstance(flight, dict) or not {'u', 'rho', 'alpha', 'beta', 'n', 'm'} <= flight.keys():
            raise ValueError(f'Missing flight or mesh parameters: {path}')
        loaded = VLM({}, 1.0, 1.0, 0.0, 0.0, 1, 1)
        loaded.apply_configuration(configuration, include_flight=True)
        with contextlib.redirect_stdout(io.StringIO()):
            loaded.load_results(str(path))
        cases.append(loaded)
    return cases


with tempfile.TemporaryDirectory() as temporary_directory:
    results_path = Path(temporary_directory) / 'case.json'
    case.save_results(str(results_path))
    restored_case = load_result_cases([results_path])[0]
    assert np.allclose(restored_case.gammas, case.gammas)
    assert np.isclose(restored_case.CL, case.CL) and np.isclose(restored_case.CD, case.CD)
print('Results saved and restored successfully in a temporary directory.')

Results saved and restored successfully in a temporary directory.


## 2. Reference Quantities, Axes, and Units

The vector post-processing defines $b=\sum b_k$, $S_{ref}=\sum b_k(c_{r,k}+c_{t,k})/2$, and $\bar c=\sum b_k(c_{r,k}^2+c_{r,k}c_{t,k}+c_{t,k}^2)/(3S_{ref})$. Here $b_k$ is the **total** span of each symmetric section. These expressions use the specified dimensions, which are not necessarily the projected area when dihedral is present.

Dynamic pressure is $q_\infty=\frac12\rho U^2$. The moment reference point `reference['point']` is the area-weighted average of the bound-segment centers for **all main-wing panels**. It should not automatically be interpreted as the quarter point of the mean aerodynamic chord or as the center of gravity.

Panels are stored internally in `(y, x, z)` order and reordered by post-processing to `(x, y, z)`: $x$ is longitudinal along the freestream when $\alpha=\beta=0$, $y$ is lateral, and $z$ is vertical. Moments follow $\mathbf M=\mathbf r\times\mathbf F$ and the right-hand rule. The coefficients `CX`, `CY`, `CZ`, `Cl`, `Cm`, and `Cn` are components in these axes, not wind axes. When comparing with other tools, check their $z$ convention, origin, and freestream direction.

In [18]:
def references(vlm):
    sections = vlm.plane['wing_sections']
    spans = np.array([section['span_fraction'] for section in sections], dtype=float)
    root = np.array([section['chord_root'] for section in sections], dtype=float)
    tip = np.array([section['chord_tip'] for section in sections], dtype=float)
    if not np.isfinite(np.r_[spans, root, tip, vlm.rho, vlm.u]).all():
        raise ValueError('Dimensions, density, and speed must be finite.')
    if np.any(spans <= 0) or np.any(root <= 0) or np.any(tip <= 0) or vlm.rho <= 0 or vlm.u <= 0:
        raise ValueError('Dimensions, density, and speed must be positive.')
    area = np.sum(spans * (root + tip) / 2)
    mean_chord = np.sum(spans * (root**2 + root * tip + tip**2) / 3) / area
    return {'S': area, 'b': spans.sum(), 'c': mean_chord,
            'q': 0.5 * vlm.rho * vlm.u**2, 'point': np.asarray(vlm.aerodynamic_coefficients['reference']['point'])}


ref = references(case)
assert np.isclose(ref['S'], 8.0) and np.isclose(ref['b'], 8.0) and np.isclose(ref['c'], 1.0)
assert np.isclose(ref['S'], case.aerodynamic_coefficients['reference']['area'])
print(f"S={ref['S']:.3f} m², b={ref['b']:.3f} m, c̄={ref['c']:.3f} m, q={ref['q']:.2f} Pa")
print('Reference point [x, y, z] in m:', ref['point'])
print(f'Panel area used for scalar CL and CD: {case.wing_area:.6f} m²')

S=8.000 m², b=8.000 m, c̄=1.000 m, q=980.00 Pa
Reference point [x, y, z] in m: [ 4.36900421e-01 -5.55111512e-17 -2.28969809e-02]
Panel area used for scalar CL and CD: 8.000000 m²


## 3. From Circulation to Forces and Moments

The tangency condition yields $\mathbf P\boldsymbol\Gamma=\mathbf w$, with $\mathbf w=-\mathbf V_\infty\cdot\mathbf n$ after accounting for the normal modified by the camber slope. `P_ij_resis` contains the normal contribution of the trailing legs and $W_i=(\mathbf P_{resis}\boldsymbol\Gamma)_i$; it is not a complete three-dimensional velocity field.

In the **scalar** path `plane()`, each panel contributes $L_i^{esc}=\rho U\Gamma_i\ell_i^{panel}$ and $D_i^{esc}=-\rho W_i\Gamma_i\ell_i^{panel}$, where $\ell_i^{panel}$ is the length calculated from the panel vertices. All surfaces are summed; the totals are divided by $q_\infty$ and the discretized main-wing area `wing_area`. The `CL` and `CD` values are not a general vector projection into wind axes, especially for configurations with a vertical stabilizer or sideslip.

In the **vector** path, the code uses the bound segment $\boldsymbol\ell_i$ and calculates

$$\mathbf F_i=\rho\Gamma_i(\mathbf V_\infty\times\boldsymbol\ell_i)-D_i^{vec}\widehat{\mathbf V}_\infty,\qquad D_i^{vec}=-\rho W_i\Gamma_i\|\boldsymbol\ell_i\|.$$

Moments are evaluated at the midpoint of each bound segment: $\mathbf M_{ref}=\sum_i(\mathbf r_i-\mathbf r_{ref})\times\mathbf F_i$. The following reconstruction reproduces the implementation to check signs and dimensions. **It is not a second independent aerodynamic solution.**

In [19]:
def panel_loads(vlm):
    nodes = np.asarray([[panel[2][0], panel[2][1]] for panel in vlm.panel_data], dtype=float)[:, :, [1, 0, 2]]
    segments = nodes[:, 1] - nodes[:, 0]
    positions = nodes.mean(axis=1)
    velocity = np.asarray(vlm.u_, dtype=float)[[1, 0, 2]]
    circulation = np.asarray(vlm.gammas, dtype=float)
    induced_normal = vlm.P_ij_resis @ circulation
    bound_force = vlm.rho * circulation[:, None] * np.cross(velocity, segments)
    induced_drag = -vlm.rho * induced_normal * circulation * np.linalg.norm(segments, axis=1)
    force = bound_force - induced_drag[:, None] * velocity / np.linalg.norm(velocity)
    point = references(vlm)['point']
    moment = np.cross(positions - point, force)
    return {'force': force, 'moment': moment, 'position': positions,
            'segment': segments, 'induced_normal': induced_normal,
            'drag': induced_drag, 'velocity': velocity}


panels = panel_loads(case)
residual = np.linalg.norm(case.P_ij @ case.gammas - case.w_i) / max(np.linalg.norm(case.w_i), np.finfo(float).eps)
assert panels['force'].shape == (len(case.panel_data), 3)
assert np.isfinite(panels['force']).all() and np.isfinite(panels['moment']).all()
assert np.allclose(panels['force'].sum(axis=0), case.aerodynamic_coefficients['force'], rtol=1e-12, atol=1e-10)
assert np.allclose(panels['moment'].sum(axis=0), case.aerodynamic_coefficients['moment'], rtol=1e-12, atol=1e-10)
print(f'Relative system residual: {residual:.3e}')
print('Force [N]:', panels['force'].sum(axis=0))
print('Moment about the reference point [N·m]:', panels['moment'].sum(axis=0))

Relative system residual: 6.874e-16
Force [N]: [ -21.02635499    0.         1934.40406909]
Moment about the reference point [N·m]: [-6.96331881e-13  3.74445897e+02 -1.29896094e-14]


### Comparison with an Equivalent Pressure

An independent physical integration of $\Delta p\,\mathbf n\,dS$ would require a differential-pressure distribution and consistent normals. The VLM solver for this case does not expose an independent $\Delta p$, and its stripwise local coefficients must not be mistaken for a pressure field. As an **algebraic** check, we define $\Delta p_i^{equiv}=L_i^{esc}/A_i$ and integrate it over the panels. This recovers the same load by construction; it does not validate the model or verify an independent pressure calculation.

In [20]:
panel_areas = np.asarray(case.panel_areas, dtype=float)
panel_lengths = np.array([np.linalg.norm(np.asarray(panel[1][0]) - np.asarray(panel[1][3])) for panel in case.panel_data])
assert panel_areas.shape == case.gammas.shape and np.all(panel_areas > 0)
scalar_lift = case.rho * case.u * case.gammas * panel_lengths
equivalent_pressure = scalar_lift / panel_areas
lift_from_equivalent_pressure = np.sum(equivalent_pressure * panel_areas)
stored_lift = sum(np.sum(distribution) for distribution in case.lift_sum.values())
assert np.isclose(lift_from_equivalent_pressure, stored_lift, rtol=1e-12, atol=1e-10)
print(f'Scalar integration: {stored_lift:.6f} N')
print(f'Integrated equivalent pressure: {lift_from_equivalent_pressure:.6f} N')
print('Algebraic difference [N]:', lift_from_equivalent_pressure - stored_lift)

Scalar integration: 1934.404069 N
Integrated equivalent pressure: 1934.404069 N
Algebraic difference [N]: 0.0


## 4. Global Coefficients and Transformation to Wind Axes

In the vector path, $C_X=F_x/(qS_{ref})$, $C_Y=F_y/(qS_{ref})$, $C_Z=F_z/(qS_{ref})$, and $(C_l,C_m,C_n)=(M_x/(qSb),M_y/(qS\bar c),M_z/(qSb))$. The reference point and dimensions must accompany the results.

For a projection independent of the scalar path, define $\hat{\mathbf e}_D=\mathbf V_\infty/\|\mathbf V_\infty\|$, $\hat{\mathbf e}_Y$ as the $y$ axis projected perpendicular to $\hat{\mathbf e}_D$, and $\hat{\mathbf e}_L=\hat{\mathbf e}_D\times\hat{\mathbf e}_Y$. Thus $C_{D,v}=-\mathbf F\cdot\hat{\mathbf e}_D/(qS)$ and $C_{L,v}=\mathbf F\cdot\hat{\mathbf e}_L/(qS)$. The exact direction of $\mathbf V_\infty$ is taken from `u_`, reordered as in the solver; no conventional external rotation of $\alpha$ and $\beta$ is assumed. The projection is shown separately from `case.CL` and `case.CD`.

In [21]:
def reconstructed_coefficients(vlm):
    ref = references(vlm)
    panels = panel_loads(vlm)
    force = panels['force'].sum(axis=0)
    moment = panels['moment'].sum(axis=0)
    scale = ref['q'] * ref['S']
    force_coefficients = force / scale
    moment_coefficients = moment / (scale * np.array([ref['b'], ref['c'], ref['b']]))
    freestream_axis = panels['velocity'] / np.linalg.norm(panels['velocity'])
    lateral_axis = np.array([0.0, 1.0, 0.0]) - freestream_axis[1] * freestream_axis
    lateral_norm = np.linalg.norm(lateral_axis)
    if lateral_norm < 1e-12:
        raise ValueError('The lateral axis cannot be defined for this freestream direction.')
    lateral_axis /= lateral_norm
    lift_axis = np.cross(freestream_axis, lateral_axis)
    return {'CX': force_coefficients[0], 'CY': force_coefficients[1], 'CZ': force_coefficients[2],
            'Cl': moment_coefficients[0], 'Cm': moment_coefficients[1], 'Cn': moment_coefficients[2],
            'CL_wind': force @ lift_axis / scale,
            'CD_wind': -force @ freestream_axis / scale}


coefficients = reconstructed_coefficients(case)
for name in ('CX', 'CY', 'CZ', 'Cl', 'Cm', 'Cn'):
    assert np.isclose(coefficients[name], getattr(case, name), rtol=1e-12, atol=1e-12)
for name, value in coefficients.items():
    print(f'{name:10s}: {value:+.7f}')
print(f'VLMPy scalar CL and CD: {case.CL:+.7f}, {case.CD:+.7f}')

CX        : -0.0026819
CY        : +0.0000000
CZ        : +0.2467352
Cl        : -0.0000000
Cm        : +0.0477610
Cn        : -0.0000000
CL_wind   : +0.2467352
CD_wind   : +0.0026819
VLMPy scalar CL and CD: +0.2467352, +0.0026819


## 5. Induced Drag and Numerical Diagnostics

Scalar `CD` comes from the wake through $-\rho W_i\Gamma_i\ell_i^{panel}$; the vector contribution uses the bound-segment length and opposes the freestream. Neither includes profile, parasitic, or wave drag. For a flat symmetric wing at zero angle of attack, the drag predicted by the model is zero: a nonzero residual is a **numerical indicator**, not a measurement of viscous drag. Mesh sensitivity is quantified later.

Far-field energy or Trefftz-plane integration is not implemented in this notebook: `P_ij_resis` provides only a normal projection of velocity, not the required field. It would therefore be incorrect to present that approach as a second independent estimate of $C_{D_i}$.

In [22]:
zero_case = solve_case(rectangular_wing, alpha_deg=0.0)
asymmetric_wing = {'wing_sections': [dict(rectangular_wing['wing_sections'][0],
    flap_start=0.6, flap_end=1.0, flap_hinge_chord=0.7,
    deflection_angle=np.deg2rad(5.0), deflection_type='antisymmetrical')]}
aileron_case = solve_case(asymmetric_wing, alpha_deg=3.0, n=16, m=4)
print(f'Scalar induced CD at 3°: {case.CD:.7f}; vector projection: {coefficients["CD_wind"]:.7f}')
print(f'CL and CD at 0°: {zero_case.CL:.3e}, {zero_case.CD:.3e}')
print('Asymmetric aileron: ' + ', '.join(f'{name}={getattr(aileron_case, name):+.6f}' for name in ('CY', 'Cl', 'Cn')))
assert np.isfinite([zero_case.CL, zero_case.CD, aileron_case.CY, aileron_case.Cl, aileron_case.Cn]).all()

Scalar induced CD at 3°: 0.0026819; vector projection: 0.0026819
CL and CD at 0°: 0.000e+00, 0.000e+00
Asymmetric aileron: CY=+0.000058, Cl=+0.031197, Cn=+0.001124


## 6. Sectional Distributions

For the main wing, `CL_locals_wing` normalizes the scalar lift of each strip by its own area. The sum of circulations per strip is shown as a loading diagnostic; the **estimated** center of pressure $x_{cp}=\sum x_iF_{z,i}/\sum F_{z,i}$ uses only the $F_z$ component and the bound-segment positions. It is not a complete center of pressure when lateral forces exist, and it becomes unstable when a strip's vertical force approaches zero.

The smoothed curve is for visualization only: it is not used to verify symmetry, extrema, or convergence. The tips deserve particular attention because actual circulation tends to zero there, but the last sample is at a strip center, not at the wingtip.

In [23]:
def wing_distributions(vlm):
    panels = panel_loads(vlm)
    count = vlm.n * vlm.m
    positions = panels['position'][:count].reshape(vlm.n, vlm.m, 3)
    forces = panels['force'][:count].reshape(vlm.n, vlm.m, 3)
    vertical_force = forces[:, :, 2].sum(axis=1)
    threshold = max(np.max(np.abs(vertical_force)) * 1e-8, np.finfo(float).eps)
    center_of_pressure = np.full(vlm.n, np.nan)
    np.divide((positions[:, :, 0] * forces[:, :, 2]).sum(axis=1), vertical_force,
              out=center_of_pressure, where=np.abs(vertical_force) > threshold)
    return {'y': positions[:, :, 1].mean(axis=1), 'cl': np.asarray(vlm.CL_locals_wing),
            'load': np.asarray(vlm.lift_sum['lift_wing']),
            'gamma': np.asarray(vlm.gammas[:count]).reshape(vlm.n, vlm.m).sum(axis=1),
            'x_cp': center_of_pressure}


def smooth_for_display(values, window=3):
    if window < 1 or window % 2 == 0:
        raise ValueError('The window must be a positive odd integer.')
    edge = window // 2
    return np.convolve(np.pad(values, (edge, edge), mode='edge'), np.ones(window) / window, mode='valid')


distribution = wing_distributions(case)
fig_distribution = make_subplots(rows=2, cols=2, subplot_titles=('Strip c_l', 'Scalar load [N/strip]', 'Sum of Γ [m²/s]', 'Estimated x_cp [m]'))
for index, key in enumerate(('cl', 'load', 'gamma', 'x_cp')):
    row, column = divmod(index, 2)
    fig_distribution.add_trace(go.Scatter(x=distribution['y'], y=distribution[key], mode='lines+markers', name=key), row=row + 1, col=column + 1)
fig_distribution.add_trace(go.Scatter(x=distribution['y'], y=smooth_for_display(distribution['cl']), mode='lines', name='Smoothed c_l', line={'dash': 'dash'}), row=1, col=1)
fig_distribution.update_layout(height=650, title='Main-wing distributions', showlegend=True)
fig_distribution.update_xaxes(title_text='y [m]')
fig_distribution.show()

## 7. Angle-of-Attack Sweep

For small angles, fit $C_L=C_{L0}+C_{L_\alpha}\alpha$, with $\alpha$ in radians and $C_{L_\alpha}$ in rad$^{-1}$. The plotted band is $\pm1.96$ times the fit residual standard deviation: it describes only the fit error for these deterministic points. It is **not** experimental uncertainty, a model confidence interval, or discretization error.

In [24]:
angles_deg = np.array([-4.0, -2.0, 0.0, 2.0, 4.0])
sweep_cases = [solve_case(rectangular_wing, angle) for angle in angles_deg]
results = [{'alpha_deg': angle, 'CL': current.CL, 'CD': current.CD,
            'CX': current.CX, 'CY': current.CY, 'CZ': current.CZ,
            'Cl': current.Cl, 'Cm': current.Cm, 'Cn': current.Cn}
           for angle, current in zip(angles_deg, sweep_cases)]
alpha_rad = np.deg2rad(angles_deg)
cl_values = np.array([row['CL'] for row in results])
slope, intercept = np.polyfit(alpha_rad, cl_values, 1)
fit = slope * alpha_rad + intercept
residual_sigma = np.sqrt(np.sum((cl_values - fit)**2) / (len(alpha_rad) - 2))
print(f'CL_alpha = {slope:.6f} rad⁻¹; CL0 = {intercept:+.3e}; residual sigma = {residual_sigma:.3e}')
print(' alpha [°]       CL         CD')
for row in results:
    print(f"{row['alpha_deg']:8.1f}  {row['CL']:+.6f}  {row['CD']:+.6f}")
fig_fit = go.Figure()
fig_fit.add_trace(go.Scatter(x=angles_deg, y=fit + 1.96 * residual_sigma, mode='lines', line={'width': 0}, showlegend=False))
fig_fit.add_trace(go.Scatter(x=angles_deg, y=fit - 1.96 * residual_sigma, mode='lines', line={'width': 0}, fill='tonexty', name='±1.96 residual σ'))
fig_fit.add_trace(go.Scatter(x=angles_deg, y=cl_values, mode='markers', name='VLMPy'))
fig_fit.add_trace(go.Scatter(x=angles_deg, y=fit, mode='lines', name='Linear fit'))
fig_fit.update_layout(title='Lift curve', xaxis_title='α [degrees]', yaxis_title='CL')
fig_fit.show()

CL_alpha = 4.718666 rad⁻¹; CL0 = +1.716e-17; residual sigma = 4.422e-04
 alpha [°]       CL         CD
    -4.0  -0.329667  +0.004779
    -2.0  -0.164228  +0.001190
     0.0  +0.000000  +0.000000
     2.0  +0.164228  +0.001190
     4.0  +0.329667  +0.004779


## 8. Independent Comparison with Lifting-Line Theory

To compare an unswept, untwisted **rectangular** wing, we independently solve Prandtl's equation with a Fourier series. The reference uses $a_0=2\pi$ rad$^{-1}$, a symmetric section with $\alpha_0=0$, and incompressible flow. For $y=-\frac b2\cos\theta$:

$$\Gamma(\theta)=2bU\sum_{k=1}^{N} A_k\sin(k\theta),\quad
\sum_{k=1}^{N} A_k\sin(k\theta_i)\left(\frac{4b}{a_0c}+\frac{k}{\sin\theta_i}\right)=\alpha.$$

Once solved, $C_L=\pi AR A_1$ and $C_{D_i}=\pi AR\sum k A_k^2$. This reference computes its own loading distribution; it does not use VLMPy's circulation. Lifting-line theory and VLM are different models, so discrepancies include differences in formulation and mesh. This comparison is **verification against theory**, not experimental validation.

Ideal elliptical loading also provides $a_{el}=2\pi AR/(AR+2)$ and $C_{D_i,el}=C_L^2/(\pi AR)$. For the rectangular wing it is only an ideal reference, not an exact solution of the case or an experimental tolerance.

**Sources:** J. G. Leishman, [Lifting Line & Finite Wing Theory](https://eaglepubs.erau.edu/introductiontoaerospaceflightvehicles/chapter/lifting-line-theory/), equations 35, 54, 80, and 89-92; NASA Glenn, [Induced Drag Coefficient](https://www1.grc.nasa.gov/beginners-guide-to-aeronautics/induced-drag-coefficient/) and [Lift Equation](https://www1.grc.nasa.gov/beginners-guide-to-aeronautics/lift-equation/).

In [25]:
def rectangular_lifting_line(span, chord, alpha_rad, modes=40, slope_2d=2 * np.pi):
    if span <= 0 or chord <= 0 or modes < 1 or slope_2d <= 0:
        raise ValueError('Span, chord, slope, and number of modes must be positive.')
    theta = np.arange(1, modes + 1) * np.pi / (modes + 1)
    indices = np.arange(1, modes + 1)
    sines = np.sin(theta[:, None] * indices[None, :])
    matrix = sines * (4 * span / (slope_2d * chord) + indices[None, :] / np.sin(theta[:, None]))
    series_coefficients = np.linalg.solve(matrix, np.full(modes, alpha_rad))
    aspect_ratio = span / chord
    return {'CL': np.pi * aspect_ratio * series_coefficients[0],
            'CDi': np.pi * aspect_ratio * np.sum(indices * series_coefficients**2),
            'modes': series_coefficients, 'condition': np.linalg.cond(matrix)}


theory = rectangular_lifting_line(ref['b'], ref['c'], np.deg2rad(3.0))
fine_theory = rectangular_lifting_line(ref['b'], ref['c'], np.deg2rad(3.0), modes=60)
lifting_line_slope = rectangular_lifting_line(ref['b'], ref['c'], 1.0)['CL']
aspect_ratio = ref['b']**2 / ref['S']
elliptical_slope = 2 * np.pi * aspect_ratio / (aspect_ratio + 2)
comparisons = [('CL at 3°', case.CL, theory['CL']), ('CDi at 3°', case.CD, theory['CDi']),
               ('CL_alpha [rad⁻¹]', slope, lifting_line_slope)]
print('Quantity                  VLMPy       Lifting line     Absolute error  Relative error')
for name, calculated, reference in comparisons:
    absolute_error = abs(calculated - reference)
    relative_error = absolute_error / abs(reference) if abs(reference) > 1e-14 else np.nan
    print(f'{name:22s} {calculated:10.6f} {reference:18.6f} {absolute_error:15.6f} {relative_error:14.2%}')
print(f'Ideal elliptical slope: {elliptical_slope:.6f} rad⁻¹')
print(f'Ideal elliptical CDi using VLMPy CL: {case.CL**2 / (np.pi * aspect_ratio):.7f}')
print(f"Lifting-line change from 40 to 60 modes: ΔCL={abs(theory['CL'] - fine_theory['CL']):.3e}, ΔCDi={abs(theory['CDi'] - fine_theory['CDi']):.3e}")
assert np.isfinite([theory['CL'], theory['CDi'], lifting_line_slope, theory['condition']]).all()

Quantity                  VLMPy       Lifting line     Absolute error  Relative error
CL at 3°                 0.246735           0.253301        0.006566          2.59%
CDi at 3°                0.002682           0.002726        0.000044          1.60%
CL_alpha [rad⁻¹]         4.718666           4.837698        0.119033          2.46%
Ideal elliptical slope: 5.026548 rad⁻¹
Ideal elliptical CDi using VLMPy CL: 0.0024223
Lifting-line change from 40 to 60 modes: ΔCL=2.304e-07, ΔCDi=1.321e-08


## 9. Symmetry, Balance, and Convergence

Symmetry of a wing without sweep, dihedral, twist, or sideslip requires mirrored loads and $C_Y$, $C_l$, and $C_n$ close to zero. The reconstructed force and moment sums and the linear residual check internal consistency, not integral fluid conservation or experimental validation.

Refine `n` and `m` separately while keeping the other dimension fixed. For three meshes with ratio $r=2$, estimate $p=\log_2|\Delta_1/\Delta_2|$ and $E_R=|\Delta_2|/(2^p-1)$ only when the increments have the same sign and decrease. This extrapolation assumes an asymptotic regime that three meshes cannot establish; otherwise, report "not estimable" without hiding oscillations.

In [26]:
def richardson_three_meshes(values):
    coarse, medium, fine = np.asarray(values, dtype=float)
    coarse_delta = medium - coarse
    fine_delta = fine - medium
    if not np.isfinite([coarse, medium, fine]).all() or coarse_delta * fine_delta <= 0 or abs(fine_delta) >= abs(coarse_delta):
        return {'order': np.nan, 'error': np.nan, 'monotonic': False}
    order = np.log2(abs(coarse_delta / fine_delta))
    return {'order': order, 'error': abs(fine_delta) / (2**order - 1), 'monotonic': True}


def mesh_study(plane, alpha_deg, meshes):
    return [{'n': n, 'm': m, 'case': solve_case(plane, alpha_deg, n=n, m=m)} for n, m in meshes]


studies = {
    'span': mesh_study(rectangular_wing, 3.0, [(8, 4), (16, 4), (32, 4)]),
    'chord': mesh_study(rectangular_wing, 3.0, [(24, 2), (24, 4), (24, 8)]),
}
for direction, rows in studies.items():
    print(f'\n{direction.capitalize()} refinement:')
    for row in rows:
        current = row['case']
        print(f"n={row['n']:2d} m={row['m']:2d}  CL={current.CL:+.7f}  CD={current.CD:+.7f}  Cm={current.Cm:+.7f}")
    for name in ('CL', 'CD', 'Cm'):
        estimate = richardson_three_meshes([getattr(row['case'], name) for row in rows])
        if estimate['monotonic']:
            print(f"  {name}: apparent p={estimate['order']:.3f}, estimated error={estimate['error']:.3e}")
        else:
            print(f'  {name}: order and error not estimable with these meshes')

load = distribution['load']
symmetry_error = np.max(np.abs(load - load[::-1])) / max(np.max(np.abs(load)), np.finfo(float).eps)
print(f'\nMaximum relative scalar-load symmetry error: {symmetry_error:.3e}')
print(f'Symmetric lateral coefficients: CY={case.CY:+.3e}, Cl={case.Cl:+.3e}, Cn={case.Cn:+.3e}')


Span refinement:
n= 8 m= 4  CL=+0.2558628  CD=+0.0026012  Cm=+0.0492502
n=16 m= 4  CL=+0.2491277  CD=+0.0026622  Cm=+0.0481994
n=32 m= 4  CL=+0.2456271  CD=+0.0026932  Cm=+0.0475179
  CL: apparent p=0.944, estimated error=3.788e-03
  CD: apparent p=0.978, estimated error=3.193e-05
  Cm: apparent p=0.625, estimated error=1.257e-03

Chord refinement:
n=24 m= 2  CL=+0.2464395  CD=+0.0029443  Cm=+0.0320040
n=24 m= 4  CL=+0.2467352  CD=+0.0026819  Cm=+0.0477610
n=24 m= 8  CL=+0.2467981  CD=+0.0025378  Cm=+0.0555509
  CL: apparent p=2.234, estimated error=1.697e-05
  CD: apparent p=0.865, estimated error=1.756e-04
  Cm: apparent p=1.016, estimated error=7.617e-03

Maximum relative scalar-load symmetry error: 2.485e-15
Symmetric lateral coefficients: CY=+0.000e+00, Cl=-1.110e-17, Cn=-2.071e-19


### Visual Comparison of Mesh Sensitivity

The following plots use the already calculated meshes and separate spanwise from chordwise refinement. Each coefficient keeps its own scale. A smooth-looking curve or an apparent Richardson order does not establish an asymptotic regime; the flap-free case, reference dimensions, and operating point are those defined above.

In [ ]:
fig_mesh_coefficients = make_subplots(
    rows=2, cols=3,
    subplot_titles=('Spanwise CL', 'Spanwise CD', 'Spanwise Cm',
                    'Chordwise CL', 'Chordwise CD', 'Chordwise Cm'),
)
for row_index, (direction, mesh_key) in enumerate((('span', 'n'), ('chord', 'm')), 1):
    rows = studies[direction]
    for column, coefficient in enumerate(('CL', 'CD', 'Cm'), 1):
        values = [getattr(row['case'], coefficient) for row in rows]
        assert np.isfinite(values).all()
        fig_mesh_coefficients.add_trace(go.Scatter(
            x=[row[mesh_key] for row in rows], y=values, mode='lines+markers', name=coefficient,
            showlegend=False,
        ), row=row_index, col=column)
        fig_mesh_coefficients.update_xaxes(title_text=f'{mesh_key} panels', row=row_index, col=column)
        fig_mesh_coefficients.update_yaxes(title_text=coefficient, row=row_index, col=column)
fig_mesh_coefficients.update_layout(title='Mesh sensitivity of global coefficients', height=650)
fig_mesh_coefficients.show()

## 10. Reproducible Report and Export

The following table separates internal verification from theoretical comparisons. This well-conditioned case uses tolerances of $10^{-10}$ for symmetry and dimensionless residuals, and 12% for the lift-curve slope, as in the regression criterion in `tests/test_convergence_and_validation.py`. These thresholds are working criteria for this example, not physical uncertainties or a guarantee of general validity. The drag discrepancy is reported without imposing an experimental acceptance threshold.

The export function is **optional**: it creates JSON, CSV, and HTML figures in a user-selected directory. It does not write files automatically and may overwrite files with those names when called.

In [27]:
def prepare_report(vlm, reference_slope, fitted_slope, symmetry_error, residual, zero_incidence_case):
    checks = []
    def record(name, category, value, limit):
        checks.append({'criterion': name, 'category': category, 'value': float(value),
                       'limit': float(limit), 'status': 'PASS' if np.isfinite(value) and value <= limit else 'FAIL'})
    record('Relative linear residual', 'Verification', residual, 1e-10)
    record('Load symmetry', 'Verification', symmetry_error, 1e-10)
    record('Maximum lateral coefficient', 'Verification', max(abs(vlm.CY), abs(vlm.Cl), abs(vlm.Cn)), 1e-10)
    record('CL at zero incidence', 'Verification', abs(zero_incidence_case.CL), 1e-10)
    record('CD at zero incidence', 'Verification', abs(zero_incidence_case.CD), 1e-10)
    record('Slope versus lifting-line theory', 'Theoretical comparison', abs(fitted_slope - reference_slope) / abs(reference_slope), 0.12)
    return checks


report = prepare_report(case, lifting_line_slope, slope, symmetry_error, residual, zero_case)
print(f'{"Criterion":39s} {"Category":23s} {"Value":>12s} {"Limit":>12s} Status')
for row in report:
    print(f"{row['criterion']:39s} {row['category']:23s} {row['value']:12.3e} {row['limit']:12.3e} {row['status']}")
print(f"CDi difference versus lifting-line theory (informational): {abs(case.CD - theory['CDi']) / abs(theory['CDi']):.2%}")
assert all(row['status'] == 'PASS' for row in report), 'One or more checks failed; inspect the report.'

fig_polar = go.Figure()
fig_polar.add_trace(go.Scatter(x=[row['CL'] for row in results], y=[row['CD'] for row in results], mode='lines+markers', name='VLMPy'))
reference_polar = [rectangular_lifting_line(ref['b'], ref['c'], angle) for angle in alpha_rad]
fig_polar.add_trace(go.Scatter(x=[row['CL'] for row in reference_polar], y=[row['CDi'] for row in reference_polar], mode='lines+markers', name='Lifting-line theory'))
fig_polar.update_layout(title='Induced-drag polar: theoretical comparison', xaxis_title='CL', yaxis_title='CDi')
fig_polar.show()

Criterion                               Category                       Value        Limit Status
Relative linear residual                Verification               6.874e-16    1.000e-10 PASS
Load symmetry                           Verification               2.485e-15    1.000e-10 PASS
Maximum lateral coefficient             Verification               1.110e-17    1.000e-10 PASS
CL at zero incidence                    Verification               0.000e+00    1.000e-10 PASS
CD at zero incidence                    Verification               0.000e+00    1.000e-10 PASS
Slope versus lifting-line theory        Theoretical comparison     2.461e-02    1.200e-01 PASS
CDi difference versus lifting-line theory (informational): 1.60%


In [28]:
import csv
import json
import tempfile


def export_report(directory, sweep_data, checks, metadata, figures):
    destination = Path(directory)
    destination.mkdir(parents=True, exist_ok=True)
    with (destination / 'report.json').open('w', encoding='utf-8') as file:
        json.dump({'metadata': metadata, 'checks': checks}, file, ensure_ascii=False, indent=2, allow_nan=False)
    with (destination / 'sweep.csv').open('w', encoding='utf-8', newline='') as file:
        writer = csv.DictWriter(file, fieldnames=list(sweep_data[0]))
        writer.writeheader()
        writer.writerows(sweep_data)
    for name, figure in figures.items():
        if Path(name).name != name or not name.endswith('.html'):
            raise ValueError('Figure names must be simple HTML filenames.')
        figure.write_html(str(destination / name), include_plotlyjs=True)
    return destination


report_metadata = {
    'geometry': rectangular_wing, 'speed_m_s': case.u, 'density_kg_m3': case.rho,
    'alpha_deg': 3.0, 'beta_deg': 0.0, 'n': case.n, 'm': case.m,
    'S_m2': float(ref['S']), 'b_m': float(ref['b']), 'c_m': float(ref['c']),
    'point_m': ref['point'].tolist(), 'numpy': np.__version__, 'python': sys.version.split()[0],
    'reference': 'Rectangular lifting-line theory, 40 modes, 2D slope 2π/rad',
}
report_figures = {'distributions.html': fig_distribution, 'fit.html': fig_fit, 'polar.html': fig_polar}
with tempfile.TemporaryDirectory() as temporary_directory:
    folder = export_report(temporary_directory, results, report, report_metadata, report_figures)
    assert json.loads((folder / 'report.json').read_text(encoding='utf-8'))['checks'] == report
    assert all((folder / name).is_file() for name in ('sweep.csv', *report_figures))
print('Export verified in a temporary directory; no permanent files were created.')

# To export permanently, select a directory and uncomment:
# export_report('coefficient_results', results, report, report_metadata, report_figures)

Export verified in a temporary directory; no permanent files were created.


## Verification: Dimensional Scaling and Moment Reference

For this steady incompressible model, changing only freestream speed or density should scale dimensional loads with $q_\infty=\frac12\rho U^2$ while leaving force and moment coefficients unchanged. This check is an implementation invariant, not an external aerodynamic validation. Moments about a shifted reference point must also obey $\mathbf M_B=\mathbf M_A-(\mathbf r_B-\mathbf r_A)\times\mathbf F$.

In [ ]:
scaling_cases = [
    solve_case(rectangular_wing, alpha_deg=3.0, speed=20.0, density=1.225),
    solve_case(rectangular_wing, alpha_deg=3.0, speed=80.0, density=1.225),
    solve_case(rectangular_wing, alpha_deg=3.0, speed=40.0, density=0.8),
]
def all_coefficients(vlm):
    return {**vlm.aerodynamic_coefficients['force_coefficients'],
            **vlm.aerodynamic_coefficients['moment_coefficients']}

baseline_coefficients = all_coefficients(case)
for scaled_case in scaling_cases:
    scaled_coefficients = all_coefficients(scaled_case)
    assert scaled_coefficients.keys() == baseline_coefficients.keys()
    assert np.allclose(list(scaled_coefficients.values()), list(baseline_coefficients.values()), rtol=2e-10, atol=1e-12)
    q_ratio = (0.5 * scaled_case.rho * scaled_case.u**2) / ref['q']
    assert np.isclose(scaled_case.CL / case.CL, 1.0, rtol=2e-10, atol=1e-12)
    dimensional_force_ratio = np.linalg.norm(scaled_case.aerodynamic_coefficients['force']) / np.linalg.norm(case.aerodynamic_coefficients['force'])
    assert np.isclose(dimensional_force_ratio, q_ratio, rtol=2e-10, atol=1e-12)
    print(f"rho={scaled_case.rho:.3f} kg/m³, U={scaled_case.u:.1f} m/s: q ratio={q_ratio:.3f}; all six vector coefficients invariant")

shift = np.array([0.37, -0.22, 0.13])
force = np.asarray(case.aerodynamic_coefficients['force'], dtype=float)
moment_at_a = np.asarray(case.aerodynamic_coefficients['moment'], dtype=float)
point_a = np.asarray(case.aerodynamic_coefficients['reference']['point'], dtype=float)
point_b = point_a + shift
moment_at_b_reconstructed = moment_at_a - np.cross(point_b - point_a, force)
moment_at_b_panels = np.cross(panels['position'] - point_b, panels['force']).sum(axis=0)
assert np.allclose(moment_at_b_panels, moment_at_b_reconstructed, rtol=1e-12, atol=1e-10)
print('PASS: translated panel moments match the rigid-reference transformation [N·m]:', moment_at_b_panels)

## Verification and Validation

Force/moment reconstruction, symmetry, residuals, and mesh studies verify internal behavior. The independent rectangular lifting-line comparison is a theoretical model comparison, not experimental validation. The pressure-equivalent reconstruction is algebraic and not an independent pressure measurement.

## References

- [NASA Glenn: Lift Equation](https://www1.grc.nasa.gov/beginners-guide-to-aeronautics/lift-equation/) and [Induced Drag Coefficient](https://www1.grc.nasa.gov/beginners-guide-to-aeronautics/induced-drag-coefficient/).
- [Leishman: Lifting-Line and Finite-Wing Theory](https://eaglepubs.erau.edu/introductiontoaerospaceflightvehicles/chapter/lifting-line-theory/), equations cited in the comparison above.

## Use Cases

Use this workflow to audit reference quantities, reconstruct panel loads, compare coefficient conventions, and inspect local and global results.

## Possible Improvements

Compare with traceable experimental data and an independent lifting-surface solver using matched axes, geometry, reference point, flow conditions, and uncertainties. Investigate chordwise sensitivity of pitching moments and quantify discretization uncertainty.

## Next Steps

Continue with [Static Aerodynamic Derivatives](07_static_aerodynamic_derivatives.ipynb), especially its discussion of a moving versus fixed moment reference.